In [10]:
from sqlalchemy import create_engine, text, inspect

In [11]:
engine = create_engine('sqlite:///../data/db/construction.db')

In [12]:
inspector = inspect(engine)

# Получить все таблицы
tables = inspector.get_table_names()
tables

['contractors', 'objects', 'progress', 'works']

In [13]:
# Для каждой таблицы вывести колонки
for table_name in tables:
    print(f"\n▶️ Таблица: {table_name}")
    columns = inspector.get_columns(table_name)
    for col in columns:
        print(f"  • {col['name']} | {col['type']} | nullable={col['nullable']}")
    
    # Внешние ключи
    fks = inspector.get_foreign_keys(table_name)
    for fk in fks:
        print(f"  ↳ FK: {fk['constrained_columns']} → {fk['referred_table']}")
    
    # Индексы
    indexes = inspector.get_indexes(table_name)
    for idx in indexes:
        print(f"  🔑 INDEX: {idx['name']} ({', '.join(idx['column_names'])})")


▶️ Таблица: contractors
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • work_id | INTEGER | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: objects
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • city | TEXT | nullable=False
  • budget | REAL | nullable=False

▶️ Таблица: progress
  • id | INTEGER | nullable=True
  • work_id | INTEGER | nullable=False
  • plan_vol | REAL | nullable=False
  • fact_vol | REAL | nullable=False
  • date | TEXT | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: works
  • id | INTEGER | nullable=True
  • object_id | INTEGER | nullable=False
  • work_type | TEXT | nullable=False
  • unit | TEXT | nullable=False
  ↳ FK: ['object_id'] → objects


In [14]:
with engine.connect() as conn:
    query = """
    SELECT * 
    FROM works
    JOIN contractors ON works.id = contractors.work_id
    JOIN objects ON works.object_id = objects.id
    JOIN progress ON works.id = progress.work_id
    WHERE 
        objects.name = 'ЖК Панорама 23' AND 
        objects.city = 'Санкт-Петербург' AND 
        progress.plan_vol > progress.fact_vol AND 
        contractors.name = 'ООО Новый Век'
    """

    result = conn.execute(text(query))

print(result.keys())
result.fetchall()

RMKeyView(['id', 'object_id', 'work_type', 'unit', 'id', 'name', 'work_id', 'id', 'name', 'city', 'budget', 'id', 'work_id', 'plan_vol', 'fact_vol', 'date'])


[(63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 277, 63, 271.79, 124.19, '2024-01-30'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 405, 63, 281.24, 185.88, '2024-10-16'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 368, 63, 412.6, 161.62, '2024-07-08'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 204, 63, 837.28, 793.2, '2024-11-21'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 181, 63, 903.98, 18.56, '2024-04-16')]

In [21]:
def get_schema_from_db(inspector) -> str:
    schema_parts = {}
    
    for table_name in inspector.get_table_names():
        columns = inspector.get_columns(table_name)
        fks = inspector.get_foreign_keys(table_name)
        schema_parts[table_name] = {
            "columns": columns,
            "foreign_keys": fks
        }
    
    return schema_parts

In [24]:
db_schemas = get_schema_from_db(inspector)

LLM

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

openrouter_api_key = os.getenv("OPEN_ROUTE_API_KEY")
if not openrouter_api_key:
    raise ValueError("OPEN_ROUTE_API_KEY environment variable not set.")

In [42]:
SYSTEM_PROMPT = f"""
Ты - ассистент, который помогает преобразовать текстовый запрос на естественном языке в SQL запрос.
У тебя есть следующая информация о структуре базы данных: {db_schemas}.
В результат выводи только в чистый SQL запрос, без объяснений и комментариев и прочего шума.
Пример:
SELECT * 
    FROM works
    JOIN contractors ON works.id = contractors.work_id
    JOIN objects ON works.object_id = objects.id
    JOIN progress ON works.id = progress.work_id
    WHERE 
        objects.name = 'ЖК Панорама 23' AND 
        objects.city = 'Санкт-Петербург' AND 
        progress.plan_vol > progress.fact_vol AND 
        contractors.name = 'ООО Новый Век'
"""

def query_llm(query: str, system_prompt: str) -> str:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_api_key,
    )

    completion = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": query
            }
        ]
    )
    return completion.choices[0].message.content

In [80]:
user_query = """
Покажи все объекты в Петербурге, для подрядчика - Новый Век, по работам связанным с покраской. 
Резульат должен включать город, название объекта, имя подрядчика, название работы, ед.изм, плановый и фактический объем работ.
"""

In [81]:
sql_query_llm = query_llm(query=user_query, system_prompt=SYSTEM_PROMPT)
sql_query_llm

"SELECT \n    objects.city, \n    objects.name, \n    contractors.name, \n    works.work_type, \n    works.unit, \n    progress.plan_vol, \n    progress.fact_vol\nFROM \n    works\nJOIN \n    contractors ON works.id = contractors.work_id\nJOIN \n    objects ON works.object_id = objects.id\nJOIN \n    progress ON works.id = progress.work_id\nWHERE \n    objects.city = 'Санкт-Петербург' \n    AND contractors.name = 'ООО Новый Век' \n    AND works.work_type = 'покраска'"

sqlglot

In [77]:
import sqlglot

In [82]:
check_sql_query = sqlglot.transpile(sql_query_llm)
check_sql_query

["SELECT objects.city, objects.name, contractors.name, works.work_type, works.unit, progress.plan_vol, progress.fact_vol FROM works JOIN contractors ON works.id = contractors.work_id JOIN objects ON works.object_id = objects.id JOIN progress ON works.id = progress.work_id WHERE objects.city = 'Санкт-Петербург' AND contractors.name = 'ООО Новый Век' AND works.work_type = 'покраска'"]

In [83]:
with engine.connect() as conn:
    query = check_sql_query[0]
    result = conn.execute(text(query))

print(result.keys())
result.fetchall()

RMKeyView(['city', 'name', 'name', 'work_type', 'unit', 'plan_vol', 'fact_vol'])


[]

In [ ]:
(['city', 'name', 'name', 'work_type', 'unit', 'plan_vol', 'fact_vol'])
[('Санкт-Петербург', 'Детский сад 36', 'ООО Новый Век', 'Монтаж перекрытий', 'шт', 548.97, 162.17),
 ('Санкт-Петербург', 'ЖК Панорама 23', 'ООО Новый Век', 'Окраска', 'м³', 271.79, 124.19)]